In [2]:
import sys
sys.path.append("/Users/ghadena/Desktop/geopol/BLINK")  

In [7]:
import torch
print(torch.__version__)

/Users/ghadena/miniconda3/envs/blink38/lib/python3.8/site-packages/torch/_subclasses/functional_tensor.py:258: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


2.4.1


In [4]:
import sys
print(sys.executable)

/Users/ghadena/miniconda3/envs/blink38/bin/python


In [6]:
!pip install torch
# print(torch.__version__)

  Using cached filelock-3.16.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 MB 3.3 MB/s eta 0:00:0000:0100:01
Using cached filelock-3.16.1-py3-none-any.whl (16 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 3.5 MB/s eta 0:00:00a 0:00:01
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


In [28]:
!pip install json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [25]:
# from blink.biencoder.zeshel_utils import load_entity_dict
from blink.main_dense import run as blink_run

In [29]:
import pandas as pd
import json
import uuid
import os
from blink.main_dense import run as blink_run

def run_blink_in_notebook(entity_df, entity_col="entity", blink_base_path=".", top_k=1):
    """
    Runs BLINK entity linking on a DataFrame column.
    
    Args:
        entity_df (pd.DataFrame): must include a column with entity mentions.
        entity_col (str): name of the column with entity strings.
        blink_base_path (str): path to BLINK repo base.
        top_k (int): how many top predictions to retrieve.
    
    Returns:
        DataFrame with new column 'linked_entity'
    """
    unique_entities = entity_df[entity_col].dropna().unique()
    blink_input = [
        {"id": i, "context_left": "", "mention": str(ent), "context_right": ""}
        for i, ent in enumerate(unique_entities)
    ]

    temp_id = str(uuid.uuid4())[:8]
    input_path = f"blink_input_{temp_id}.jsonl"
    output_path = f"blink_output_{temp_id}.jsonl"

    with open(input_path, "w") as f:
        for item in blink_input:
            f.write(json.dumps(item) + "\n")

    # Run BLINK
    args = {
        "test_mentions": input_path,
        "biencoder_model": f"{blink_base_path}/models/biencoder/params.txt",
        "biencoder_config": f"{blink_base_path}/models/biencoder/bert_base",
        "faiss_index": f"{blink_base_path}/models/faiss/index.faiss",
        "entity_catalogue": f"{blink_base_path}/models/entity.jsonl",
        "entity_encoding": f"{blink_base_path}/models/all_entities.pkl",
        "top_k": top_k,
        "output_path": output_path,
    }
    blink_run(args)

    # Load output
    with open(output_path, "r") as f:
        results = [json.loads(line) for line in f]

    # Map predictions
    result_map = {
        blink_input[r["id"]]["mention"]: r["pred_triples"][0][0] for r in results
    }
    
    entity_df["linked_entity"] = entity_df[entity_col].map(result_map)

    # Clean up
    os.remove(input_path)
    os.remove(output_path)

    return entity_df

In [30]:
df_entities = pd.read_csv('/Users/ghadena/Desktop/geopol/data/processed/entities.csv')

In [39]:
import subprocess
import os

def run_blink_in_notebook(entity_df, entity_col="entity", blink_base_path=".", top_k=1):
    import json, uuid

    unique_entities = entity_df[entity_col].dropna().unique()
    blink_input = [
        {"id": i, "context_left": "", "mention": str(ent), "context_right": ""}
        for i, ent in enumerate(unique_entities)
    ]

    temp_id = str(uuid.uuid4())[:8]
    input_path = f"blink_input_{temp_id}.jsonl"
    output_path = f"blink_output_{temp_id}.jsonl"

    with open(input_path, "w") as f:
        for item in blink_input:
            f.write(json.dumps(item) + "\n")

    # Set PYTHONPATH so the blink module is recognized
    env = os.environ.copy()
    env["PYTHONPATH"] = blink_base_path + ":" + env.get("PYTHONPATH", "")

    command = [
        "python", os.path.join(blink_base_path, "blink/main_dense.py"),
        "--test_mentions", input_path,
        "--biencoder_model", os.path.join(blink_base_path, "models/biencoder/params.txt"),
        "--biencoder_config", os.path.join(blink_base_path, "models/biencoder"),
        "--faiss_index", os.path.join(blink_base_path, "models/faiss/index.faiss"),
        "--entity_catalogue", os.path.join(blink_base_path, "models/entity.jsonl"),
        "--entity_encoding", os.path.join(blink_base_path, "models/all_entities.pkl"),
        "--top_k", str(top_k),
        "--output_path", output_path
    ]

    try:
        subprocess.run(command, check=True, env=env)
    except subprocess.CalledProcessError as e:
        print("⚠️ BLINK failed:", e)
        return entity_df

    with open(output_path, "r") as f:
        results = [json.loads(line) for line in f]

    result_map = {
        blink_input[r["id"]]["mention"]: r["pred_triples"][0][0]
        for r in results
    }

    entity_df["linked_entity"] = entity_df[entity_col].map(result_map)

    os.remove(input_path)
    os.remove(output_path)

    return entity_df

In [40]:
df_linked = run_blink_in_notebook(df_entities, entity_col="entity", blink_base_path="/Users/ghadena/Desktop/geopol/BLINK")

05/22/2025 23:30:50 - INFO - Blink -   loading biencoder model


Traceback (most recent call last):
  File "/Users/ghadena/Desktop/geopol/BLINK/blink/main_dense.py", line 692, in <module>
    models = load_models(args, logger)
  File "/Users/ghadena/Desktop/geopol/BLINK/blink/main_dense.py", line 294, in load_models
    with open(args.biencoder_config) as json_file:
FileNotFoundError: [Errno 2] No such file or directory: '/Users/ghadena/Desktop/geopol/BLINK/models/biencoder'


⚠️ BLINK failed: Command '['python', '/Users/ghadena/Desktop/geopol/BLINK/blink/main_dense.py', '--test_mentions', 'blink_input_0a36753c.jsonl', '--biencoder_model', '/Users/ghadena/Desktop/geopol/BLINK/models/biencoder/params.txt', '--biencoder_config', '/Users/ghadena/Desktop/geopol/BLINK/models/biencoder', '--faiss_index', '/Users/ghadena/Desktop/geopol/BLINK/models/faiss/index.faiss', '--entity_catalogue', '/Users/ghadena/Desktop/geopol/BLINK/models/entity.jsonl', '--entity_encoding', '/Users/ghadena/Desktop/geopol/BLINK/models/all_entities.pkl', '--top_k', '1', '--output_path', 'blink_output_0a36753c.jsonl']' returned non-zero exit status 1.


In [41]:
df_linked.head()

,Unnamed: 0,url,date,entity,entity_type
0,0,https://www.nytimes.com/live/2024/09/09/us/har...,2024-09-09,Donald Trump,people
1,1,https://www.nytimes.com/live/2024/09/09/us/har...,2024-09-09,Kamala Harris,people
2,2,https://www.nytimes.com/live/2024/09/09/us/har...,2024-09-09,Marco Rubio,people
3,3,https://www.nytimes.com/live/2024/09/09/us/har...,2024-09-09,Ted Cruz,people
4,4,https://www.nytimes.com/live/2024/09/09/us/har...,2024-09-09,Hillary Clinton,people


In [42]:
with open("blink_output_262c4568.jsonl", "r") as f:
    for line in f:
        print(json.loads(line))

IsADirectoryError: [Errno 21] Is a directory: 'blink_output_262c4568.jsonl'

## new try 

In [43]:
!pip install transformers torch

In [46]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("Wikinews/wikidata-entity-linker")
model = AutoModelForTokenClassification.from_pretrained("Wikinews/wikidata-entity-linker")

linker = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text = "Angela Merkel and Emmanuel Macron met in Berlin."

results = linker(text)

for r in results:
    print(f"🔗 {r['word']} → {r['entity_group']} (score: {r['score']:.2f})")

OSError: Wikinews/wikidata-entity-linker is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

# spacy


In [47]:
import spacy
import wikipediaapi
import requests

# Load SpaCy NER model
nlp = spacy.load("en_core_web_lg")

# Setup Wikipedia API
wiki = wikipediaapi.Wikipedia("en")

# Optional: link entity to Wikidata QID
def get_wikidata_qid(title):
    url = f"https://en.wikipedia.org/w/api.php?action=query&prop=pageprops&format=json&titles={title}"
    try:
        r = requests.get(url).json()
        page = next(iter(r["query"]["pages"].values()))
        return page.get("pageprops", {}).get("wikibase_item", None)
    except:
        return None

# Entity linking pipeline
def link_entities(text):
    doc = nlp(text)
    linked = []
    for ent in doc.ents:
        wiki_page = wiki.page(ent.text)
        if wiki_page.exists():
            qid = get_wikidata_qid(wiki_page.title)
            linked.append({
                "mention": ent.text,
                "label": ent.label_,
                "wiki_title": wiki_page.title,
                "url": wiki_page.fullurl,
                "qid": qid
            })
    return linked

ModuleNotFoundError: No module named 'spacy'

In [49]:
!pip install "spacy==3.8.2" --prefer-binary


  Using cached spacy-3.8.2.tar.gz (1.3 MB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [68 lines of output]
      Ignoring numpy: markers 'python_version >= "3.9"' don't match your environment
        Using cached setuptools-75.3.2-py3-none-any.whl.metadata (6.9 kB)
        Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
        Using cached preshed-3.0.9-cp38-cp38-macosx_11_0_arm64.whl.metadata (2.2 kB)
        Using cached thinc-8.3.2.tar.gz (193 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'error'
        error: subprocess-exited-with-error
      
        × pip subprocess to install build dependencies did not run successfully.
        │ exit code: 1
        ╰─> [38 lines of output]
            Ignoring numpy: markers 'python_version >= "3.9"' don't match your envir

In [ ]:
df_linked = run_blink_in_notebook(df_entities, entity_col="entity", blink_base_path="/Users/ghadena/Desktop/geopol/BLINK")

## Spacy 2

In [1]:
!pip install --upgrade pip setuptools wheel

#python -m spacy download en_core_web_lg

  Using cached pip-25.1.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached setuptools-80.8.0-py3-none-any.whl.metadata (6.6 kB)
Using cached pip-25.1.1-py3-none-any.whl (1.8 MB)
Using cached setuptools-80.8.0-py3-none-any.whl (1.2 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 78.1.1
    Uninstalling setuptools-78.1.1:
      Successfully uninstalled setuptools-78.1.1
  Attempting uninstall: pip━━━━━━━━━━━━━━━━━━━━━ 0/2 [setuptools]
    Found existing installation: pip 25.1━━━ 0/2 [setuptools]
    Uninstalling pip-25.1:━━━━━━━━━━━━━━━━━━ 0/2 [setuptools]
      Successfully uninstalled pip-25.1━━━━━ 0/2 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pip]1/2 [pip]


In [2]:
!pip install spacy wikipedia-api requests

  Using cached spacy-3.8.6-cp310-cp310-macosx_11_0_arm64.whl.metadata (27 kB)
  Using cached wikipedia_api-0.8.1-py3-none-any.whl
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.13-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.2 kB)
  Using cached cymem-2.0.11-cp310-cp310-macosx_11_0_arm64.whl.metadata (8.5 kB)
  Using cached preshed-3.0.9-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.2 kB)
  Using cached thinc-8.3.6-cp310-cp310-macosx_11_0_arm64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.1-cp310-cp310-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached typer-0.15.4-py3-none-any.whl.metadata (15 kB)
  Usin

In [3]:
import spacy.cli
spacy.cli.download("en_core_web_lg")

     ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.1/400.7 MB 626.5 kB/s eta 0:07:35


error: incomplete-download

× Download failed because not enough bytes were received (116.1 MB/400.7 MB)
╰─> URL: https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl

note: This is an issue with network connectivity, not pip.
hint: Consider using --resume-retries to enable download resumption.


SystemExit: 1

/Users/ghadena/Desktop/geopol/.conda/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.0 MB/s eta 0:00:0000:0100:06


In [7]:
!pip install pandas


  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 1.3 MB/s eta 0:00:00a 0:00:01m
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [5]:
import spacy
nlp = spacy.load("en_core_web_lg")


In [8]:
install pandas as pd 

df_entities = pd.read_csv('/Users/ghadena/Desktop/geopol/data/processed/entities.csv')

SyntaxError: invalid syntax (1873226248.py, line 1)

In [11]:
import pandas as pd
df_entities = pd.read_csv('/Users/ghadena/Desktop/geopol/data/processed/entities.csv')

In [14]:
import spacy
import wikipediaapi
import requests

# Load SpaCy large model
nlp = spacy.load("en_core_web_lg")

# Set up Wikipedia API
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="ghadena-entity-linker/1.0 (https://github.com/ghadena)"
)

# Helper: get Wikidata QID
def get_wikidata_qid(title):
    try:
        url = f"https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "prop": "pageprops",
            "format": "json",
            "titles": title
        }
        r = requests.get(url, params=params).json()
        page = next(iter(r["query"]["pages"].values()))
        return page.get("pageprops", {}).get("wikibase_item", None)
    except:
        return None

In [15]:
def link_entities(text):
    doc = nlp(text)
    linked = []

    for ent in doc.ents:
        wiki_page = wiki.page(ent.text)
        if wiki_page.exists():
            qid = get_wikidata_qid(wiki_page.title)
            linked.append({
                "mention": ent.text,
                "label": ent.label_,
                "wiki_title": wiki_page.title,
                "url": wiki_page.fullurl,
                "qid": qid
            })

    return linked

In [16]:
text = "Angela Merkel met with Emmanuel Macron and Netanyahu in Berlin."

results = link_entities(text)

for r in results:
    print(f"🔗 {r['mention']} ({r['label']})")
    print(f"→ Wikipedia: {r['wiki_title']}")
    print(f"→ Wikidata QID: {r['qid']}")
    print(f"→ URL: {r['url']}")
    print("---")

🔗 Angela Merkel (PERSON)
→ Wikipedia: Angela Merkel
→ Wikidata QID: Q567
→ URL: https://en.wikipedia.org/wiki/Angela_Merkel
---
🔗 Emmanuel Macron (PERSON)
→ Wikipedia: Emmanuel Macron
→ Wikidata QID: Q3052772
→ URL: https://en.wikipedia.org/wiki/Emmanuel_Macron
---
🔗 Netanyahu (PERSON)
→ Wikipedia: Benjamin Netanyahu
→ Wikidata QID: Q43723
→ URL: https://en.wikipedia.org/wiki/Benjamin_Netanyahu
---
🔗 Berlin (GPE)
→ Wikipedia: Berlin
→ Wikidata QID: Q64
→ URL: https://en.wikipedia.org/wiki/Berlin
---


In [17]:
df_entities["linked_entities"] = df_entities["text"].apply(link_entities)

KeyError: 'text'

In [20]:
import wikipediaapi
import requests
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()  

# Set up Wikipedia API
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="ghadena-entity-linker/1.0 (https://github.com/ghadena)"
)

# Helper: Get Wikidata QID for a Wikipedia title
def get_wikidata_qid(title):
    try:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "prop": "pageprops",
            "format": "json",
            "titles": title
        }
        r = requests.get(url, params=params).json()
        page = next(iter(r["query"]["pages"].values()))
        return page.get("pageprops", {}).get("wikibase_item", None)
    except:
        return None

# Main function: clean + resolve each entity
def resolve_entity(entity):
    page = wiki.page(entity)
    if page.exists():
        qid = get_wikidata_qid(page.title)
        return pd.Series({
            "canonical_title": page.title,
            "wiki_url": page.fullurl,
            "wikidata_qid": qid
        })
    else:
        return pd.Series({
            "canonical_title": None,
            "wiki_url": None,
            "wikidata_qid": None
        })

In [19]:
resolved = df_entities["entity"].apply(resolve_entity)
df_entities = pd.concat([df_entities, resolved], axis=1)

KeyboardInterrupt: 

In [ ]:
df_entities

In [21]:
# Take 5 unique entities for testing
df_test = df_entities["entity"].dropna().drop_duplicates().sample(5, random_state=42).to_frame()

In [25]:
resolved = df_test["entity"].progress_apply(resolve_entity)
df_test = pd.concat([df_test, resolved], axis=1)

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [23]:
import wikipediaapi
import requests
import pandas as pd
from tqdm.notebook import tqdm

# Setup
tqdm.pandas()
wiki = wikipediaapi.Wikipedia("en", user_agent="ghadena-entity-linker/1.0 (https://github.com/ghadena)")

def get_wikidata_qid(title):
    try:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "prop": "pageprops",
            "format": "json",
            "titles": title
        }
        r = requests.get(url, params=params).json()
        page = next(iter(r["query"]["pages"].values()))
        return page.get("pageprops", {}).get("wikibase_item", None)
    except:
        return None

def resolve_entity(entity):
    page = wiki.page(entity)
    if page.exists():
        qid = get_wikidata_qid(page.title)
        return pd.Series({
            "canonical_title": page.title,
            "wiki_url": page.fullurl,
            "wikidata_qid": qid
        })
    else:
        return pd.Series({
            "canonical_title": None,
            "wiki_url": None,
            "wikidata_qid": None
        })

# Mini test
df_test = df_entities["entity"].dropna().drop_duplicates().sample(5, random_state=42).to_frame()
resolved = df_test["entity"].progress_apply(resolve_entity)
df_test = pd.concat([df_test, resolved], axis=1)

TypeError: Wikipedia.__init__() got multiple values for argument 'user_agent'

In [24]:
!pip install ipywidgets

  Using cached ipywidgets-8.1.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.14-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.7-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl (216 kB)
Using cached widgetsnbextension-4.0.14-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [29]:
from tqdm import tqdm

# Create an empty list to collect rows
resolved_rows = []

for entity in tqdm(df_entities["entity"], desc="Resolving entities"):
    resolved_rows.append(resolve_entity(entity))

# Convert list of Series to a DataFrame
resolved_df = pd.DataFrame(resolved_rows)

# Merge back with original test data
df_entities = pd.concat([df_entities.reset_index(drop=True), resolved_df], axis=1)


Resolving entities:   9%|▉         | 1352/15080 [21:00<3:33:19,  1.07it/s] 


ReadTimeout: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=10.0)

In [27]:
df_test

,entity,canonical_title,wiki_url,wikidata_qid
0,Joe Kennedy III,Joe Kennedy III,https://en.wikipedia.org/wiki/Joe_Kennedy_III,Q1707784
1,Peter Ricketts,Peter Ricketts,https://en.wikipedia.org/wiki/Peter_Ricketts,Q4394632
2,Hassan Hussein Abdullah,None,None,None
3,Wells Fargo Investment Institute,None,None,None
4,Balázs Orbán,Balázs Orbán,https://en.wikipedia.org/wiki/Bal%C3%A1zs_Orb%...,Q657730
